In [2]:
# importing libraries

import numpy as np
import pandas as pd

from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

from sklearn.ensemble import RandomForestClassifier

In [3]:
train_df = pd.read_pickle("../data/train.pkl")
test_df = pd.read_pickle("../data/test.pkl")

In [4]:
x_train = train_df.drop(columns=["isFraud","TransactionDT"])
y_train = train_df["isFraud"]

x_test = test_df.drop(columns=["TransactionDT"])

In [5]:
# Catboost

In [6]:
categorical_cols = x_train.select_dtypes(include=["object"]).columns.tolist()

In [7]:
x_train_cat = x_train.copy()
x_test_cat = x_test.copy()

x_train_cat[categorical_cols] = x_train_cat[categorical_cols].fillna("Missing")
x_test_cat[categorical_cols] = x_test_cat[categorical_cols].fillna("Missing")

In [8]:
catboost_final = CatBoostClassifier(iterations=300 , learning_rate=0.05 , depth=6 , loss_function="Logloss" , eval_metric="AUC",
                                random_seed=42 , verbose=100)

catboost_final.fit(x_train_cat , y_train , cat_features=categorical_cols)

0:	total: 8.28s	remaining: 41m 17s
100:	total: 11m 22s	remaining: 22m 24s
200:	total: 23m	remaining: 11m 19s
299:	total: 34m 28s	remaining: 0us


CatBoostClassifier(depth=6, eval_metric='AUC', iterations=300, learning_rate=0.05, loss_function='Logloss', random_seed=42, verbose=100)

In [9]:
catboost_test_pred = catboost_final.predict_proba(x_test_cat)[:,1]

catboost_test_output =pd.DataFrame({
    "TransactionID" : test_df["TransactionID"] , 
    "catboost_test" : catboost_test_pred
})

catboost_test_output.to_csv("../outputs/catboost_test_predictions.csv" , index=False)

In [10]:
%whos

Variable                 Type                  Data/Info
--------------------------------------------------------
CatBoostClassifier       type                  <class 'catboost.core.CatBoostClassifier'>
ColumnTransformer        ABCMeta               <class 'sklearn.compose._<...>ormer.ColumnTransformer'>
LGBMClassifier           type                  <class 'lightgbm.sklearn.LGBMClassifier'>
OneHotEncoder            type                  <class 'sklearn.preproces<...>_encoders.OneHotEncoder'>
Pipeline                 ABCMeta               <class 'sklearn.pipeline.Pipeline'>
RandomForestClassifier   ABCMeta               <class 'sklearn.ensemble.<...>.RandomForestClassifier'>
SimpleImputer            type                  <class 'sklearn.impute._base.SimpleImputer'>
XGBClassifier            type                  <class 'xgboost.sklearn.XGBClassifier'>
catboost_final           CatBoostClassifier    CatBoostClassifier(depth=<...>dom_seed=42, verbose=100)
catboost_test_output     DataFram

In [11]:
import joblib
import os

os.makedirs("../models/",exist_ok=True)
catboost_final.save_model("../models/catboost_final.cbm")

print(" Catboost model saved successfully. ")

 Catboost model saved successfully. 


In [12]:
joblib.dump(list(x_train_cat.columns) , "../models/catboost_feature_columns.pkl")
joblib.dump(list(categorical_cols) , "../models/catboost_categorical_columns.pkl")

print(" Catboost Feature metadata saved successfully.")

 Catboost Feature metadata saved successfully.
